In [ ]:
import json
import os
import uuid
from datetime import datetime, timezone

import boto3
import requests
from dotenv import load_dotenv


load_dotenv(
    os.path.expanduser("~/stock-market-streaming/.env")
)

FINNHUB_API_KEY = os.getenv("FINNHUB_API_KEY")

S3_BUCKET = "kafka-spark-stock-project-kris"

S3_BASE_PATH = (
    "stock-market/bronze/rest/basic_financials"
)

SYMBOLS = [
    "AAPL",
    "MSFT",
    "NVDA",
    "TSLA",
    "AMZN",
]

s3 = boto3.client("s3")


def fetch_basic_financials(symbol):

    url = "https://finnhub.io/api/v1/stock/metric"

    params = {
        "symbol": symbol,
        "metric": "all",
        "token": FINNHUB_API_KEY,
    }

    response = requests.get(
        url,
        params=params,
        timeout=15,
    )

    response.raise_for_status()

    raw_data = response.json()

    if not raw_data:
        raise ValueError(
            f"Empty financial data returned for {symbol}"
        )

    return {
        "symbol": symbol,
        "metric_type": raw_data.get("metricType"),

        # Keep full metric dictionary in Bronze
        "metrics": raw_data.get("metric", {}),

        # Preserve full Finnhub response
        "raw_financials": raw_data,
    }


def upload_to_s3(records):

    now = datetime.now(timezone.utc)
    batch_id = str(uuid.uuid4())

    payload = {
        "schema_version": "1.0",
        "dataset": "basic_financials",
        "source": "finnhub",
        "batch_id": batch_id,
        "collected_at_utc": now.isoformat(),
        "record_count": len(records),
        "records": records,
    }

    s3_key = (
        f"{S3_BASE_PATH}/"
        f"year={now.year}/"
        f"month={now.month}/"
        f"day={now.day}/"
        f"basic_financials_"
        f"{now.strftime('%Y%m%dT%H%M%S')}_"
        f"{batch_id}.json"
    )

    s3.put_object(
        Bucket=S3_BUCKET,
        Key=s3_key,
        Body=json.dumps(payload).encode("utf-8"),
        ContentType="application/json",
    )

    return s3_key


def main():

    if not FINNHUB_API_KEY:
        raise RuntimeError(
            "FINNHUB_API_KEY not found."
        )

    records = []

    for symbol in SYMBOLS:

        try:

            financials = fetch_basic_financials(symbol)

            records.append(financials)

            metric_count = len(
                financials["metrics"]
            )

            print(
                f"{symbol}: "
                f"{metric_count} metrics collected"
            )

        except Exception as error:

            print(
                f"Failed to fetch {symbol}: {error}"
            )

    if not records:
        raise RuntimeError(
            "No basic financial data collected."
        )

    s3_key = upload_to_s3(records)

    print()
    print("=" * 60)
    print(f"Collected companies: {len(records)}")
    print("Basic financials batch uploaded to S3")
    print(f"s3://{S3_BUCKET}/{s3_key}")
    print("=" * 60)


if __name__ == "__main__":
    main()